# ACXNet++: Attention-Augmented Hybrid Deep Learning for Cross-Task EEG Mental Workload Estimation

Full reproduction pipeline for **ACXNet++** (Sundaravadivel & Abinaya, extending Abinaya & Dinakaran 2025's
ACXNet), on the **STEW (Simultaneous Task EEG Workload)** dataset.

Pipeline: raw EEG → band-pass filter + normalize → 2s/1s-step windowing → parallel branches
**(autoencoder → CNN → multi-head self-attention)** and **(36-d handcrafted features)** → fused →
**XGBoost** classifier, under **subject-level 5-fold `StratifiedGroupKFold` cross-validation**,
with paired Wilcoxon signed-rank significance testing.

## Before you run this
1. Get the STEW dataset from IEEE DataPort: https://doi.org/10.21227/44r8-ya50
   (`STEW_Dataset.zip` — 96 files named `subNN_lo.txt` / `subNN_hi.txt`, one row per EEG
   sample, 14 whitespace-separated channel columns — plus `ratings.txt`, CSV rows of
   `subject_id, rating_lo, rating_hi`; ratings for subjects 5, 24, 42 are missing.)
2. Upload `STEW_Dataset.zip` (and `ratings.txt`, if not already inside the zip) to this
   Colab session — either drag into the Files pane, or run the upload cell below.
3. Runtime → Change runtime type → GPU is optional (paper's pipeline is CPU-only and runs
   in ~35-40 min end-to-end on CPU; a GPU will make the CNN/attention/autoencoder stages faster).

All hyperparameters below are taken directly from the paper's Table 2 / Sections 3.2-3.11.

In [ ]:
!pip install -q xgboost scikit-learn scipy numpy torch pandas matplotlib seaborn

In [ ]:
import os
import io
import glob
import json
import zipfile
import warnings
import random

import numpy as np
import pandas as pd
import scipy.signal as sps
from scipy.stats import skew, kurtosis, wilcoxon
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, confusion_matrix,
)
from sklearn.preprocessing import StandardScaler

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset

import xgboost as xgb
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

# ----------------------------------------------------------------------------
# Global config (Table 2 / Section 3.11)
# ----------------------------------------------------------------------------
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

FS = 128                      # Hz, Emotiv EPOC sampling rate
N_CHANNELS = 14
WINDOW_SEC = 2
STEP_SEC = 1
WINDOW_LEN = WINDOW_SEC * FS   # 256 samples
STEP_LEN = STEP_SEC * FS       # 128 samples (50% overlap)

BANDPASS_LOW, BANDPASS_HIGH = 1.0, 40.0
FIR_NUMTAPS = 101

AE_HIDDEN = [512, 256]
AE_LATENT = 128
AE_LR = 1e-3
AE_BATCH = 128
AE_EPOCHS = 25

CNN_LR = 1e-3
CNN_WD = 1e-5
CNN_BATCH = 128
CNN_EPOCHS = 30
N_ATTN_HEADS = 4

XGB_PARAMS = dict(
    n_estimators=150,
    learning_rate=0.05,
    max_depth=4,
    subsample=0.7,
    colsample_bytree=0.7,
    min_child_weight=5,
    reg_lambda=2.0,
    reg_alpha=0.5,
    eval_metric="logloss",
    early_stopping_rounds=15,
    tree_method="hist",
)

N_FOLDS = 5
VAL_HOLDOUT_FRAC = 0.15   # of each fold's training subjects, held out for early stopping

BAND_DEFS = {
    "delta": (0.5, 4.0),
    "theta": (4.0, 8.0),
    "alpha": (8.0, 12.0),
    "beta": (12.0, 30.0),
    "low_gamma": (30.0, 40.0),
}
APEN_CHANNELS = [0, 3, 6, 9, 12]  # 5 representative channels spread across the montage

print(f"Device: {DEVICE}")

## 1. Data loading

Expects a directory containing `subNN_lo.txt` / `subNN_hi.txt` (raw EEG, one row per sample,
14 whitespace-separated channels) for each subject, and `ratings.txt` (CSV: `subject, rating_lo,
rating_hi`, missing rows for subjects without valid ratings).

In [ ]:
# Uncomment to upload interactively in Colab:
# from google.colab import files
# uploaded = files.upload()   # select STEW_Dataset.zip (and ratings.txt if separate)

DATA_ROOT = "/content/STEW_Dataset"   # change if your extracted path differs
ZIP_PATH = "/content/STEW_Dataset.zip"

if not os.path.isdir(DATA_ROOT) and os.path.exists(ZIP_PATH):
    with zipfile.ZipFile(ZIP_PATH, "r") as zf:
        zf.extractall("/content/")
    # STEW_Dataset.zip commonly extracts flat or into a subfolder; locate it
    candidates = [p for p in glob.glob("/content/**/sub*_lo.txt", recursive=True)]
    if candidates:
        DATA_ROOT = os.path.dirname(candidates[0])

print(f"DATA_ROOT = {DATA_ROOT}")
assert os.path.isdir(DATA_ROOT), (
    "STEW data not found. Upload STEW_Dataset.zip to /content/ (or set DATA_ROOT/ZIP_PATH "
    "above) before continuing — see the markdown cell at the top of this notebook."
)

In [ ]:
def find_ratings_file(root):
    for cand in [os.path.join(root, "ratings.txt"), "/content/ratings.txt"]:
        if os.path.exists(cand):
            return cand
    hits = glob.glob(os.path.join(root, "**", "ratings.txt"), recursive=True)
    if hits:
        return hits[0]
    raise FileNotFoundError("ratings.txt not found — upload it alongside the STEW EEG files.")


def load_ratings(root):
    """ratings.txt: CSV rows of `subject, rating_lo, rating_hi`.
    Subjects 5, 24, 42 have no valid rating and are dropped, leaving the 45 usable subjects
    used throughout this study (Section 3.1)."""
    path = find_ratings_file(root)
    df = pd.read_csv(path, header=None, names=["subject", "rating_lo", "rating_hi"])
    df["subject"] = df["subject"].astype(int)
    df = df.dropna(subset=["rating_lo", "rating_hi"]).reset_index(drop=True)
    return df.set_index("subject")


def load_raw_eeg(root, subject_id, condition):
    """condition in {'lo', 'hi'}. Returns array of shape (n_samples, 14)."""
    fname = f"sub{subject_id:02d}_{condition}.txt"
    hits = glob.glob(os.path.join(root, "**", fname), recursive=True)
    if not hits:
        hits = [os.path.join(root, fname)]
    path = hits[0]
    arr = np.loadtxt(path)
    if arr.ndim == 1:
        arr = arr.reshape(-1, N_CHANNELS)
    return arr[:, :N_CHANNELS].astype(np.float64)


ratings_df = load_ratings(DATA_ROOT)
USABLE_SUBJECTS = sorted(ratings_df.index.tolist())
print(f"Usable subjects (valid ratings): {len(USABLE_SUBJECTS)}")
assert len(USABLE_SUBJECTS) == 45, (
    f"Expected 45 usable subjects per the paper, found {len(USABLE_SUBJECTS)} — "
    "check ratings.txt / missing-subject handling."
)

## 2. Preprocessing (Section 3.2)

Step 1 — FIR band-pass filter (101 taps, 1-40 Hz, Hamming window, zero-phase via `filtfilt`).
Step 2 — per-channel min-max normalization to [0, 1].
Step 3 — windowing (2 s / 256 samples, 1 s / 128-sample step, 50% overlap).

(This study does not apply ICA artifact rejection — see Section 3.2 / Section 6 of the paper
for why: it is not reproducible without the original researchers' manual component labels.)

In [ ]:
_FIR_COEFFS = sps.firwin(
    FIR_NUMTAPS, [BANDPASS_LOW, BANDPASS_HIGH], pass_zero=False, fs=FS, window="hamming"
)


def bandpass_filter(x):
    """x: (n_samples, n_channels) -> zero-phase FIR band-pass filtered, same shape."""
    return sps.filtfilt(_FIR_COEFFS, [1.0], x, axis=0)


def minmax_normalize(x):
    """Per-channel min-max normalization to [0, 1]."""
    x_min = x.min(axis=0, keepdims=True)
    x_max = x.max(axis=0, keepdims=True)
    denom = np.where((x_max - x_min) < 1e-12, 1.0, x_max - x_min)
    return (x - x_min) / denom


def window_signal(x, win_len=WINDOW_LEN, step=STEP_LEN):
    """x: (n_samples, n_channels) -> (n_windows, win_len, n_channels)."""
    n_samples = x.shape[0]
    starts = list(range(0, n_samples - win_len + 1, step))
    return np.stack([x[s:s + win_len] for s in starts], axis=0)


def preprocess_subject_condition(root, subject_id, condition):
    raw = load_raw_eeg(root, subject_id, condition)
    filtered = bandpass_filter(raw)
    normed = minmax_normalize(filtered)
    windows = window_signal(normed)   # (n_windows, 256, 14)
    return windows


# Quick sanity check on one subject/condition
_test_windows = preprocess_subject_condition(DATA_ROOT, USABLE_SUBJECTS[0], "lo")
print(f"Windows per subject-condition recording: {_test_windows.shape[0]} "
      f"(paper reports 149), window shape: {_test_windows.shape[1:]}")

## 3. Labeling protocol (Section 3.3)

Median split computed **independently within each condition**: No-Task (`lo`) median = 1
(>1 → high-workload), SIMKAP (`hi`) median = 7 (>=7 → high-workload). Every window inherits
its subject's single condition-level label.

In [ ]:
def compute_subject_labels(ratings_df, condition):
    col = f"rating_{condition}"
    median = ratings_df[col].median()
    if condition == "lo":
        labels = (ratings_df[col] > median).astype(int)   # high-workload if rating > median
    else:
        labels = (ratings_df[col] >= median).astype(int)  # high-workload if rating >= median
    print(f"[{condition}] median={median}, high={labels.sum()}, low={(labels == 0).sum()}")
    return labels


LABELS_LO = compute_subject_labels(ratings_df, "lo")
LABELS_HI = compute_subject_labels(ratings_df, "hi")

## 4. Handcrafted feature extraction (Section 3.4)

18 base statistics — 6 temporal, 8 spectral, 2 nonlinear-dynamics, 2 "top-contributor" stats
(AR(1) coefficient, wavelet-detail-std proxy) — each aggregated as **mean and std across
channels**, giving 36 features per 2-second window. ApEn is computed only on 5 representative
channels for computational tractability (~13,400 windows total) and its mean/std are taken
across those 5 channels rather than all 14.

In [ ]:
def zero_crossing_rate(x):
    return np.mean(np.diff(np.sign(x)) != 0)


def spectral_features(x, fs=FS):
    """x: 1-D signal. Returns dict of PSD-based features via Hann-windowed FFT."""
    n = len(x)
    win = np.hanning(n)
    xw = x * win
    spec = np.fft.rfft(xw)
    psd = (np.abs(spec) ** 2) / (fs * np.sum(win ** 2))
    freqs = np.fft.rfftfreq(n, d=1.0 / fs)

    mean_psd = psd.mean()
    peak_freq = freqs[np.argmax(psd)]

    psd_norm = psd / (psd.sum() + 1e-12)
    spectral_entropy = -np.sum(psd_norm * np.log2(psd_norm + 1e-12)) / np.log2(len(psd_norm))

    band_powers = {}
    for name, (lo, hi) in BAND_DEFS.items():
        mask = (freqs >= lo) & (freqs < hi)
        band_powers[name] = psd[mask].mean() if mask.any() else 0.0

    return mean_psd, peak_freq, spectral_entropy, band_powers


def approximate_entropy(x, m=2, r_frac=0.2):
    """Standard ApEn (Pincus 1991)."""
    x = np.asarray(x, dtype=np.float64)
    N = len(x)
    r = r_frac * np.std(x)
    if r < 1e-12:
        return 0.0

    def _phi(m):
        templates = np.array([x[i:i + m] for i in range(N - m + 1)])
        dists = np.max(np.abs(templates[:, None, :] - templates[None, :, :]), axis=2)
        C = np.sum(dists <= r, axis=1) / (N - m + 1)
        return np.sum(np.log(C + 1e-300)) / (N - m + 1)

    return _phi(m) - _phi(m + 1)


def hurst_exponent_rs(x):
    """Single-scale rescaled-range (R/S) Hurst exponent estimate."""
    x = np.asarray(x, dtype=np.float64)
    N = len(x)
    mean_x = x.mean()
    dev = np.cumsum(x - mean_x)
    R = dev.max() - dev.min()
    S = x.std()
    if S < 1e-12 or R < 1e-12:
        return 0.5
    rs = R / S
    return np.log(rs + 1e-12) / np.log(N)


def ar1_coefficient(x):
    """First-order autoregressive coefficient via least-squares on consecutive samples."""
    x0, x1 = x[:-1], x[1:]
    denom = np.dot(x0, x0)
    if denom < 1e-12:
        return 0.0
    return float(np.dot(x0, x1) / denom)


def wavelet_detail_std_proxy(x):
    """Std of the first difference — proportional to finest-scale Haar detail-coefficient energy."""
    return float(np.std(np.diff(x)))


def extract_window_features(window):
    """window: (256, 14) -> 36-d handcrafted feature vector."""
    n_ch = window.shape[1]

    temporal = np.zeros((n_ch, 6))
    spectral = np.zeros((n_ch, 8))
    ar1_wav = np.zeros((n_ch, 2))

    apen_vals = []

    for ch in range(n_ch):
        sig = window[:, ch]

        # Temporal (6)
        mean_amp = sig.mean()
        rms_amp = np.sqrt(np.mean(sig ** 2))
        std_amp = sig.std()
        zcr = zero_crossing_rate(sig)
        sk = skew(sig)
        ku = kurtosis(sig)
        temporal[ch] = [mean_amp, rms_amp, std_amp, zcr, sk, ku]

        # Spectral (8)
        mean_psd, peak_freq, sp_ent, bands = spectral_features(sig)
        spectral[ch] = [
            mean_psd, peak_freq, sp_ent,
            bands["delta"], bands["theta"], bands["alpha"], bands["beta"], bands["low_gamma"],
        ]

        # AR(1) + wavelet-detail-std proxy (2)
        ar1_wav[ch] = [ar1_coefficient(sig), wavelet_detail_std_proxy(sig)]

        if ch in APEN_CHANNELS:
            apen_vals.append(approximate_entropy(sig))

    hurst_vals = np.array([hurst_exponent_rs(window[:, ch]) for ch in range(n_ch)])
    apen_vals = np.array(apen_vals)

    base_stats = []
    # 6 temporal + 8 spectral + 2 (AR1, wavelet) = 16, each mean/std across 14 channels -> 32
    for block in (temporal, spectral, ar1_wav):
        base_stats.append(block.mean(axis=0))
        base_stats.append(block.std(axis=0))
    # Hurst: mean/std across all 14 channels -> 2
    base_stats.append(np.array([hurst_vals.mean()]))
    base_stats.append(np.array([hurst_vals.std()]))
    # ApEn: mean/std across the 5 representative channels only -> 2
    base_stats.append(np.array([apen_vals.mean()]))
    base_stats.append(np.array([apen_vals.std()]))

    feat = np.concatenate(base_stats)
    assert feat.shape[0] == 36, f"expected 36 handcrafted features, got {feat.shape[0]}"
    return feat.astype(np.float32)

## 5. Build per-condition window datasets (raw + handcrafted features + subject/label arrays)

Run once per condition; cached to disk (`.npz`) so re-runs of the notebook skip the
~16-minute handcrafted-feature extraction pass (dominated by approximate entropy).

In [ ]:
def build_condition_dataset(root, condition, ratings_df, labels_by_subject, cache_dir="/content/cache"):
    os.makedirs(cache_dir, exist_ok=True)
    cache_path = os.path.join(cache_dir, f"stew_{condition}.npz")
    if os.path.exists(cache_path):
        d = np.load(cache_path)
        return d["windows"], d["handcrafted"], d["subject_ids"], d["labels"]

    all_windows, all_feats, all_subjects, all_labels = [], [], [], []
    for subj in USABLE_SUBJECTS:
        windows = preprocess_subject_condition(root, subj, condition)   # (n_win, 256, 14)
        label = int(labels_by_subject.loc[subj])
        for w in windows:
            all_windows.append(w)
            all_feats.append(extract_window_features(w))
            all_subjects.append(subj)
            all_labels.append(label)

    windows_arr = np.stack(all_windows).astype(np.float32)          # (N, 256, 14)
    feats_arr = np.stack(all_feats).astype(np.float32)              # (N, 36)
    subjects_arr = np.array(all_subjects, dtype=np.int32)
    labels_arr = np.array(all_labels, dtype=np.int32)

    np.savez_compressed(
        cache_path, windows=windows_arr, handcrafted=feats_arr,
        subject_ids=subjects_arr, labels=labels_arr,
    )
    return windows_arr, feats_arr, subjects_arr, labels_arr


print("Building No-Task (lo) dataset ...")
WINDOWS_LO, HAND_LO, SUBJ_LO, Y_LO = build_condition_dataset(DATA_ROOT, "lo", ratings_df, LABELS_LO)
print(f"  {WINDOWS_LO.shape[0]} windows, {len(np.unique(SUBJ_LO))} subjects")

print("Building SIMKAP (hi) dataset ...")
WINDOWS_HI, HAND_HI, SUBJ_HI, Y_HI = build_condition_dataset(DATA_ROOT, "hi", ratings_df, LABELS_HI)
print(f"  {WINDOWS_HI.shape[0]} windows, {len(np.unique(SUBJ_HI))} subjects")

CONDITION_DATA = {
    "lo": dict(windows=WINDOWS_LO, handcrafted=HAND_LO, subjects=SUBJ_LO, labels=Y_LO),
    "hi": dict(windows=WINDOWS_HI, handcrafted=HAND_HI, subjects=SUBJ_HI, labels=Y_HI),
}

## 6. Autoencoder (Section 3.5)

Fully-connected 3584 -> 512 -> 256 -> 128 encoder (ReLU), mirrored decoder with sigmoid output.
Trained to minimize MSE reconstruction loss with Adam, per-fold, on training-fold windows
from **both conditions combined** (never sees validation/test-fold subject data).

In [ ]:
class FCAutoencoder(nn.Module):
    def __init__(self, in_dim=WINDOW_LEN * N_CHANNELS, hidden=AE_HIDDEN, latent=AE_LATENT):
        super().__init__()
        h1, h2 = hidden
        self.encoder = nn.Sequential(
            nn.Linear(in_dim, h1), nn.ReLU(),
            nn.Linear(h1, h2), nn.ReLU(),
            nn.Linear(h2, latent), nn.ReLU(),
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent, h2), nn.ReLU(),
            nn.Linear(h2, h1), nn.ReLU(),
            nn.Linear(h1, in_dim), nn.Sigmoid(),
        )

    def forward(self, x):
        z = self.encoder(x)
        x_hat = self.decoder(z)
        return x_hat, z


class FlatWindowDataset(Dataset):
    """Flattens (256, 14) windows to 3584-d vectors for the autoencoder."""
    def __init__(self, windows):
        self.windows = windows

    def __len__(self):
        return len(self.windows)

    def __getitem__(self, idx):
        return torch.from_numpy(self.windows[idx].reshape(-1)).float()


def train_autoencoder(train_windows, val_windows, epochs=AE_EPOCHS, verbose=False):
    model = FCAutoencoder().to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=AE_LR)
    train_loader = DataLoader(FlatWindowDataset(train_windows), batch_size=AE_BATCH, shuffle=True)
    val_loader = DataLoader(FlatWindowDataset(val_windows), batch_size=AE_BATCH, shuffle=False)

    history = []
    for epoch in range(epochs):
        model.train()
        train_loss = 0.0
        for xb in train_loader:
            xb = xb.to(DEVICE)
            opt.zero_grad()
            x_hat, _ = model(xb)
            loss = F.mse_loss(x_hat, xb)
            loss.backward()
            opt.step()
            train_loss += loss.item() * xb.size(0)
        train_loss /= len(train_loader.dataset)

        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for xb in val_loader:
                xb = xb.to(DEVICE)
                x_hat, _ = model(xb)
                val_loss += F.mse_loss(x_hat, xb).item() * xb.size(0)
        val_loss /= max(len(val_loader.dataset), 1)
        history.append((train_loss, val_loss))
        if verbose:
            print(f"  AE epoch {epoch + 1}/{epochs}  train_mse={train_loss:.4f}  val_mse={val_loss:.4f}")

    return model, history


@torch.no_grad()
def encode_windows(ae_model, windows, batch_size=512):
    ae_model.eval()
    loader = DataLoader(FlatWindowDataset(windows), batch_size=batch_size, shuffle=False)
    latents = []
    for xb in loader:
        xb = xb.to(DEVICE)
        _, z = ae_model(xb)
        latents.append(z.cpu().numpy())
    return np.concatenate(latents, axis=0)   # (N, 128)

## 7. CNN backbone + multi-head self-attention (Sections 3.6-3.7)

128-d latent -> conv block 1 (32 filters, k=5, BN, ReLU, maxpool/2) -> length 64, 32 ch ->
conv block 2 (64 filters, k=5, BN, ReLU, maxpool/2) -> length 32, 64 ch. The 32 positions are
treated as tokens (64-d each); optionally passed through 4-head scaled dot-product
self-attention with a residual + LayerNorm; then pooled **over the channel/embedding axis**
(mean across the 64 filter responses at each of the 32 positions) to give the 32-dimensional
feature vector used for classification (matching the `cnn_attn_0 .. cnn_attn_31` feature
names reported in Section 4.8's feature-importance analysis). A linear+sigmoid auxiliary head
on top of this 32-d vector supplies the supervised training signal during backbone
pretraining only; final classification always uses XGBoost on the extracted features.

In [ ]:
class CNNAttentionBackbone(nn.Module):
    def __init__(self, latent_dim=AE_LATENT, use_attention=True, n_heads=N_ATTN_HEADS):
        super().__init__()
        self.use_attention = use_attention

        self.conv1 = nn.Conv1d(1, 32, kernel_size=5, padding=2)
        self.bn1 = nn.BatchNorm1d(32)
        self.pool1 = nn.MaxPool1d(2)

        self.conv2 = nn.Conv1d(32, 64, kernel_size=5, padding=2)
        self.bn2 = nn.BatchNorm1d(64)
        self.pool2 = nn.MaxPool1d(2)

        if use_attention:
            self.attn = nn.MultiheadAttention(
                embed_dim=64, num_heads=n_heads, batch_first=True
            )
            self.layernorm = nn.LayerNorm(64)

        self.aux_head = nn.Linear(32, 1)   # pretraining-only auxiliary classification head

    def forward(self, latent):
        # latent: (batch, 128)
        x = latent.unsqueeze(1)                      # (batch, 1, 128)
        x = self.pool1(F.relu(self.bn1(self.conv1(x))))   # (batch, 32, 64)
        x = self.pool2(F.relu(self.bn2(self.conv2(x))))   # (batch, 64, 32)

        tokens = x.transpose(1, 2)                   # (batch, 32, 64) -- 32 tokens, 64-d each

        if self.use_attention:
            attn_out, attn_weights = self.attn(tokens, tokens, tokens, need_weights=True,
                                                average_attn_weights=True)
            tokens = self.layernorm(tokens + attn_out)
        else:
            attn_weights = None

        feature = tokens.mean(dim=-1)                 # (batch, 32) -- pool over channel/embed dim
        aux_logit = self.aux_head(feature).squeeze(-1)
        return feature, aux_logit, attn_weights


class LatentDataset(Dataset):
    def __init__(self, latents, labels):
        self.latents = latents
        self.labels = labels

    def __len__(self):
        return len(self.latents)

    def __getitem__(self, idx):
        return (torch.from_numpy(self.latents[idx]).float(),
                torch.tensor(self.labels[idx]).float())


def train_cnn_backbone(train_latents, train_labels, val_latents, val_labels,
                        use_attention, epochs=CNN_EPOCHS, verbose=False):
    model = CNNAttentionBackbone(use_attention=use_attention).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=CNN_LR, weight_decay=CNN_WD)

    train_loader = DataLoader(LatentDataset(train_latents, train_labels),
                               batch_size=CNN_BATCH, shuffle=True)
    val_loader = DataLoader(LatentDataset(val_latents, val_labels),
                             batch_size=CNN_BATCH, shuffle=False)

    best_val_acc, best_state = -1.0, None
    for epoch in range(epochs):
        model.train()
        for xb, yb in train_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad()
            _, logit, _ = model(xb)
            loss = F.binary_cross_entropy_with_logits(logit, yb)
            loss.backward()
            opt.step()

        model.eval()
        correct, total = 0, 0
        with torch.no_grad():
            for xb, yb in val_loader:
                xb, yb = xb.to(DEVICE), yb.to(DEVICE)
                _, logit, _ = model(xb)
                pred = (torch.sigmoid(logit) > 0.5).float()
                correct += (pred == yb).sum().item()
                total += yb.size(0)
        val_acc = correct / max(total, 1)
        if verbose:
            print(f"  CNN(attn={use_attention}) epoch {epoch + 1}/{epochs}  val_acc={val_acc:.4f}")
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_state = {k: v.clone() for k, v in model.state_dict().items()}

    if best_state is not None:
        model.load_state_dict(best_state)
    return model, best_val_acc


@torch.no_grad()
def extract_cnn_features(model, latents, batch_size=512):
    model.eval()
    loader = DataLoader(
        torch.utils.data.TensorDataset(torch.from_numpy(latents).float()),
        batch_size=batch_size, shuffle=False,
    )
    feats = []
    for (xb,) in loader:
        xb = xb.to(DEVICE)
        f, _, _ = model(xb)
        feats.append(f.cpu().numpy())
    return np.concatenate(feats, axis=0)   # (N, 32)

## 8. XGBoost classifier + metrics (Section 3.8, Table 2)

In [ ]:
def compute_metrics(y_true, y_pred, y_proba):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    tnr = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    recall = recall_score(y_true, y_pred, zero_division=0)
    return dict(
        accuracy=accuracy_score(y_true, y_pred),
        precision=precision_score(y_true, y_pred, zero_division=0),
        recall_tpr=recall,
        f1=f1_score(y_true, y_pred, zero_division=0),
        tnr=tnr,
        far=1.0 - tnr,
        frr=1.0 - recall,
        auc=roc_auc_score(y_true, y_proba) if len(np.unique(y_true)) > 1 else float("nan"),
    )


def train_xgb_and_eval(X_subtrain, y_subtrain, X_val, y_val, X_test, y_test, seed_offset=0):
    params = dict(XGB_PARAMS)
    params["random_state"] = SEED + seed_offset
    clf = xgb.XGBClassifier(**params)
    clf.fit(X_subtrain, y_subtrain, eval_set=[(X_val, y_val)], verbose=False)

    train_pred = clf.predict(X_subtrain)
    train_acc = accuracy_score(y_subtrain, train_pred)

    test_pred = clf.predict(X_test)
    test_proba = clf.predict_proba(X_test)[:, 1]
    metrics = compute_metrics(y_test, test_pred, test_proba)
    metrics["train_accuracy"] = train_acc
    return metrics, clf, test_proba

## 9. Fold-splitting helpers

In [ ]:
def split_train_val_subjects(train_subjects, subject_label_lookup, frac=VAL_HOLDOUT_FRAC, seed=SEED):
    """Stratified subject-level split (by subject's condition label) for early-stopping."""
    from sklearn.model_selection import train_test_split
    labels = np.array([subject_label_lookup.loc[s] for s in train_subjects])
    try:
        subtrain, val = train_test_split(
            train_subjects, test_size=frac, random_state=seed, stratify=labels
        )
    except ValueError:
        subtrain, val = train_test_split(train_subjects, test_size=frac, random_state=seed)
    return np.array(subtrain), np.array(val)

## 10. Main cross-validation loop (Section 3.9)

For each condition (`lo`, `hi`) independently: 5-fold `StratifiedGroupKFold` by subject ID.
Within each fold: 15% of training subjects held out for autoencoder / CNN(+attention) /
XGBoost early stopping. The autoencoder is trained per fold on windows from **both
conditions combined**, restricted to that fold's training subjects; the CNN(+attention)
backbones and all six XGBoost configurations are trained on the current condition only.
Every fold's predictions and out-of-fold probabilities are cached for later ROC / confusion
matrix / feature-importance / attention-visualization analysis.

In [ ]:
CONFIG_NAMES = [
    "handcrafted", "latent", "cnn_noattn", "cnn_noattn+handcrafted",
    "cnn_attn", "cnn_attn+handcrafted",
]

LABELS_BY_SUBJECT = {"lo": LABELS_LO, "hi": LABELS_HI}

full_results = {}          # mirrors ACXNet_plusplus_full_results.json
oof_store = {}              # {condition: {config: {"y_true":..., "y_proba":..., "y_pred":...}}}
representative_fold_artifacts = {}   # for feature-importance / attention-weight plots


def run_condition(condition, verbose=True):
    data = CONDITION_DATA[condition]
    windows, handcrafted, subjects, labels = (
        data["windows"], data["handcrafted"], data["subjects"], data["labels"]
    )
    other_condition = "hi" if condition == "lo" else "lo"
    other_data = CONDITION_DATA[other_condition]

    sgkf = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
    splits = list(sgkf.split(X=np.zeros(len(labels)), y=labels, groups=subjects))

    fold_metrics_per_config = {name: [] for name in CONFIG_NAMES}
    oof_per_config = {name: {"y_true": [], "y_proba": [], "idx": []} for name in CONFIG_NAMES}

    for fold_idx, (train_win_idx, test_win_idx) in enumerate(splits):
        train_subjects = np.unique(subjects[train_win_idx])
        test_subjects = sorted(np.unique(subjects[test_win_idx]).tolist())

        subtrain_subj, val_subj = split_train_val_subjects(
            train_subjects, LABELS_BY_SUBJECT[condition], seed=SEED + fold_idx
        )

        subtrain_mask = np.isin(subjects, subtrain_subj)
        val_mask = np.isin(subjects, val_subj)
        test_mask = np.isin(subjects, test_subjects)

        if verbose:
            print(f"\n[{condition}] Fold {fold_idx}: "
                  f"{subtrain_mask.sum()} subtrain / {val_mask.sum()} val / {test_mask.sum()} test windows "
                  f"({len(subtrain_subj)}/{len(val_subj)}/{len(test_subjects)} subjects)")

        # ---- Autoencoder: both conditions combined, this fold's subtrain/val subjects only ----
        ae_train_windows = np.concatenate([
            windows[np.isin(subjects, subtrain_subj)],
            other_data["windows"][np.isin(other_data["subjects"], subtrain_subj)],
        ], axis=0)
        ae_val_windows = np.concatenate([
            windows[np.isin(subjects, val_subj)],
            other_data["windows"][np.isin(other_data["subjects"], val_subj)],
        ], axis=0)
        ae_model, ae_history = train_autoencoder(ae_train_windows, ae_val_windows, verbose=False)
        if verbose:
            tr, va = ae_history[-1]
            print(f"  Autoencoder final: train_mse={tr:.4f} val_mse={va:.4f}")

        # ---- Encode this condition's windows with the fold-specific autoencoder ----
        latents_all = encode_windows(ae_model, windows)

        # ---- CNN backbones (this condition only) ----
        cnn_noattn, _ = train_cnn_backbone(
            latents_all[subtrain_mask], labels[subtrain_mask],
            latents_all[val_mask], labels[val_mask], use_attention=False, verbose=False,
        )
        cnn_attn, _ = train_cnn_backbone(
            latents_all[subtrain_mask], labels[subtrain_mask],
            latents_all[val_mask], labels[val_mask], use_attention=True, verbose=False,
        )

        feat_cnn_noattn = extract_cnn_features(cnn_noattn, latents_all)
        feat_cnn_attn = extract_cnn_features(cnn_attn, latents_all)

        feature_banks = {
            "handcrafted": handcrafted,
            "latent": latents_all,
            "cnn_noattn": feat_cnn_noattn,
            "cnn_noattn+handcrafted": np.concatenate([feat_cnn_noattn, handcrafted], axis=1),
            "cnn_attn": feat_cnn_attn,
            "cnn_attn+handcrafted": np.concatenate([feat_cnn_attn, handcrafted], axis=1),
        }

        for config_name, X_all in feature_banks.items():
            X_subtrain, y_subtrain = X_all[subtrain_mask], labels[subtrain_mask]
            X_val, y_val = X_all[val_mask], labels[val_mask]
            X_test, y_test = X_all[test_mask], labels[test_mask]

            metrics, clf, test_proba = train_xgb_and_eval(
                X_subtrain, y_subtrain, X_val, y_val, X_test, y_test, seed_offset=fold_idx
            )
            metrics["fold"] = fold_idx
            metrics["test_subjects"] = test_subjects
            fold_metrics_per_config[config_name].append(metrics)

            oof_per_config[config_name]["y_true"].append(y_test)
            oof_per_config[config_name]["y_proba"].append(test_proba)
            oof_per_config[config_name]["idx"].append(np.where(test_mask)[0])

            if verbose:
                print(f"  [{config_name:24s}] acc={metrics['accuracy']:.4f} "
                      f"f1={metrics['f1']:.4f} auc={metrics['auc']:.4f}")

            # Stash artifacts from the middle fold of the full ACXNet++ config for later plots
            if config_name == "cnn_attn+handcrafted" and fold_idx == N_FOLDS // 2:
                representative_fold_artifacts[condition] = dict(
                    clf=clf, feature_names=(
                        [f"cnn_attn_{i}" for i in range(feat_cnn_attn.shape[1])]
                        + [f"handcrafted_{i}" for i in range(handcrafted.shape[1])]
                    ),
                    cnn_attn_model=cnn_attn, ae_model=ae_model,
                    latents_test=latents_all[test_mask][:2],
                )

    # ---- Aggregate per config ----
    for config_name in CONFIG_NAMES:
        fm = fold_metrics_per_config[config_name]
        key = f"{condition}__{config_name}"
        agg = {}
        for metric in ["accuracy", "precision", "recall_tpr", "f1", "tnr", "far", "frr",
                        "auc", "train_accuracy"]:
            vals = np.array([f[metric] for f in fm], dtype=np.float64)
            agg[f"{metric}_mean"] = float(np.nanmean(vals))
            agg[f"{metric}_std"] = float(np.nanstd(vals))
        agg["fold_metrics"] = fm
        agg["fold_accuracies"] = [f["accuracy"] for f in fm]
        agg["fold_f1s"] = [f["f1"] for f in fm]
        agg["fold_aucs"] = [f["auc"] for f in fm]
        full_results[key] = agg

    oof_store[condition] = oof_per_config
    return fold_metrics_per_config


for condition in ["lo", "hi"]:
    print(f"\n{'=' * 70}\nRunning condition: {condition}\n{'=' * 70}")
    run_condition(condition)

## 11. Statistical significance testing (Section 3.10)

Paired Wilcoxon signed-rank tests across the 5 fold-level accuracy values, for the same
four comparisons per condition used in the paper's Tables 5-6.

In [ ]:
SIGNIFICANCE_COMPARISONS = [
    ("cnn_attn", "cnn_noattn", "Attention vs no-attention (CNN features alone)"),
    ("cnn_attn+handcrafted", "cnn_noattn+handcrafted",
     "Attention vs no-attention (fused w/ handcrafted)"),
    ("cnn_attn+handcrafted", "handcrafted", "Full model vs handcrafted-only baseline"),
    ("cnn_attn+handcrafted", "latent", "Full model vs raw autoencoder latent baseline"),
]

significance_tests = {}
for condition in ["lo", "hi"]:
    for config_a, config_b, desc in SIGNIFICANCE_COMPARISONS:
        acc_a = np.array(full_results[f"{condition}__{config_a}"]["fold_accuracies"])
        acc_b = np.array(full_results[f"{condition}__{config_b}"]["fold_accuracies"])
        diffs = acc_a - acc_b
        if np.allclose(diffs, 0):
            stat, p = 0.0, 1.0
        else:
            stat, p = wilcoxon(acc_a, acc_b, zero_method="wilcox", alternative="two-sided")
        key = f"{condition}__{config_a}_vs_{config_b}"
        significance_tests[key] = dict(
            desc=desc, statistic=float(stat), pvalue=float(p),
            mean_a=float(acc_a.mean()), mean_b=float(acc_b.mean()),
        )
        print(f"{key:65s} p={p:.4f}  ({acc_a.mean():.3f} vs {acc_b.mean():.3f})")

## 12. Save results

In [ ]:
def _json_default(o):
    if isinstance(o, (np.integer,)):
        return int(o)
    if isinstance(o, (np.floating,)):
        return float(o)
    if isinstance(o, np.ndarray):
        return o.tolist()
    raise TypeError(f"Not JSON serializable: {type(o)}")

os.makedirs("/content/outputs", exist_ok=True)
with open("/content/outputs/ACXNet_plusplus_full_results.json", "w") as f:
    json.dump(full_results, f, indent=2, default=_json_default)
with open("/content/outputs/ACXNet_plusplus_significance_tests.json", "w") as f:
    json.dump(significance_tests, f, indent=2, default=_json_default)

print("Saved to /content/outputs/")

## 13. Results tables (Tables 3-4)

In [ ]:
def results_table(condition):
    label = "No-Task" if condition == "lo" else "SIMKAP"
    rows = []
    for config in CONFIG_NAMES:
        r = full_results[f"{condition}__{config}"]
        rows.append({
            "Configuration": config,
            "Accuracy (%)": f"{r['accuracy_mean']*100:.2f} ± {r['accuracy_std']*100:.2f}",
            "Precision (%)": f"{r['precision_mean']*100:.2f} ± {r['precision_std']*100:.2f}",
            "Recall/TPR (%)": f"{r['recall_tpr_mean']*100:.2f} ± {r['recall_tpr_std']*100:.2f}",
            "F1 (%)": f"{r['f1_mean']*100:.2f} ± {r['f1_std']*100:.2f}",
            "TNR (%)": f"{r['tnr_mean']*100:.2f} ± {r['tnr_std']*100:.2f}",
            "FAR (%)": f"{r['far_mean']*100:.2f} ± {r['far_std']*100:.2f}",
            "FRR (%)": f"{r['frr_mean']*100:.2f} ± {r['frr_std']*100:.2f}",
            "AUC": f"{r['auc_mean']:.3f} ± {r['auc_std']:.3f}",
        })
    df = pd.DataFrame(rows).set_index("Configuration")
    print(f"\n=== {label} condition ===")
    return df


display(results_table("lo"))
display(results_table("hi"))

## 14. Ablation bar charts (Figures 4-5)

In [ ]:
def plot_ablation(condition, ax):
    label = "No-Task" if condition == "lo" else "SIMKAP"
    means = [full_results[f"{condition}__{c}"]["accuracy_mean"] * 100 for c in CONFIG_NAMES]
    stds = [full_results[f"{condition}__{c}"]["accuracy_std"] * 100 for c in CONFIG_NAMES]
    colors = ["#999999", "#999999", "#4C72B0", "#4C72B0", "#DD8452", "#DD8452"]
    ax.bar(CONFIG_NAMES, means, yerr=stds, capsize=4, color=colors)
    ax.set_title(f"Ablation — {label} condition")
    ax.set_ylabel("Accuracy (%)")
    ax.set_xticklabels(CONFIG_NAMES, rotation=45, ha="right")
    ax.set_ylim(0, 100)


fig, axes = plt.subplots(1, 2, figsize=(14, 5))
plot_ablation("lo", axes[0])
plot_ablation("hi", axes[1])
plt.tight_layout()
plt.savefig("/content/outputs/ablation_study.png", dpi=150)
plt.show()

## 15. Aggregate out-of-fold ROC curves (Figure 6)

In [ ]:
def aggregate_oof(condition, config_name):
    d = oof_store[condition][config_name]
    y_true = np.concatenate(d["y_true"])
    y_proba = np.concatenate(d["y_proba"])
    return y_true, y_proba


fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
for ax, condition in zip(axes, ["lo", "hi"]):
    label = "No-Task" if condition == "lo" else "SIMKAP"
    for config_name, style in [
        ("cnn_noattn", "--"), ("cnn_attn", "-"), ("cnn_attn+handcrafted", "-.")
    ]:
        y_true, y_proba = aggregate_oof(condition, config_name)
        fpr, tpr, _ = roc_curve(y_true, y_proba)
        auc_val = roc_auc_score(y_true, y_proba)
        ax.plot(fpr, tpr, style, label=f"{config_name} (AUC={auc_val:.3f})")
    ax.plot([0, 1], [0, 1], "k:", alpha=0.4)
    ax.set_title(f"Out-of-fold ROC — {label}")
    ax.set_xlabel("False Positive Rate")
    ax.set_ylabel("True Positive Rate")
    ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig("/content/outputs/roc_curves.png", dpi=150)
plt.show()

## 16. Aggregate confusion matrices, full ACXNet++ model (Figure 7)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
for ax, condition in zip(axes, ["lo", "hi"]):
    label = "No-Task" if condition == "lo" else "SIMKAP"
    y_true, y_proba = aggregate_oof(condition, "cnn_attn+handcrafted")
    y_pred = (y_proba > 0.5).astype(int)
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    cm_pct = cm / cm.sum(axis=1, keepdims=True) * 100
    im = ax.imshow(cm_pct, cmap="Blues", vmin=0, vmax=100)
    for i in range(2):
        for j in range(2):
            ax.text(j, i, f"{cm[i, j]}\n({cm_pct[i, j]:.1f}%)", ha="center", va="center")
    ax.set_xticks([0, 1]); ax.set_xticklabels(["Low", "High"])
    ax.set_yticks([0, 1]); ax.set_yticklabels(["Low", "High"])
    ax.set_xlabel("Predicted"); ax.set_ylabel("True")
    ax.set_title(f"Confusion matrix (OOF) — {label}\nFull ACXNet++ model")
plt.tight_layout()
plt.savefig("/content/outputs/confusion_matrices.png", dpi=150)
plt.show()

## 17. Per-fold stability (Figure 8)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
representative = ["handcrafted", "latent", "cnn_noattn", "cnn_attn", "cnn_attn+handcrafted"]
for ax, condition in zip(axes, ["lo", "hi"]):
    label = "No-Task" if condition == "lo" else "SIMKAP"
    data_to_plot = [full_results[f"{condition}__{c}"]["fold_accuracies"] for c in representative]
    ax.boxplot(data_to_plot, labels=representative)
    for i, vals in enumerate(data_to_plot):
        ax.scatter([i + 1] * len(vals), vals, alpha=0.6, color="black", s=15)
    ax.set_title(f"Per-fold accuracy — {label}")
    ax.set_xticklabels(representative, rotation=45, ha="right")
    ax.set_ylabel("Accuracy")
plt.tight_layout()
plt.savefig("/content/outputs/per_fold_stability.png", dpi=150)
plt.show()

## 18. XGBoost feature importance, full model (Figure 9)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, condition in zip(axes, ["lo", "hi"]):
    label = "No-Task" if condition == "lo" else "SIMKAP"
    art = representative_fold_artifacts.get(condition)
    if art is None:
        ax.set_title(f"No artifact captured for {label}")
        continue
    clf, names = art["clf"], art["feature_names"]
    importances = clf.feature_importances_
    order = np.argsort(importances)[::-1][:15]
    ax.barh([names[i] for i in order][::-1], importances[order][::-1])
    ax.set_title(f"Top-15 feature importances — {label}\n(full ACXNet++ model, representative fold)")
plt.tight_layout()
plt.savefig("/content/outputs/feature_importance.png", dpi=150)
plt.show()

## 19. Attention weight visualization (Figure 10, SIMKAP condition)

In [ ]:
if "hi" in representative_fold_artifacts:
    art = representative_fold_artifacts["hi"]
    model, latents_examples = art["cnn_attn_model"], art["latents_test"]
    model.eval()
    with torch.no_grad():
        xb = torch.from_numpy(latents_examples).float().to(DEVICE)
        _, _, attn_weights = model(xb)   # (n_examples, 32, 32), averaged over heads
    attn_weights = attn_weights.cpu().numpy()

    fig, axes = plt.subplots(1, min(2, attn_weights.shape[0]), figsize=(9, 4))
    if attn_weights.shape[0] == 1:
        axes = [axes]
    titles = ["Example window A", "Example window B"]
    for i, ax in enumerate(axes):
        im = ax.imshow(attn_weights[i], cmap="viridis")
        ax.set_title(titles[i] if i < len(titles) else f"Example {i}")
        ax.set_xlabel("Key position"); ax.set_ylabel("Query position")
    plt.tight_layout()
    plt.savefig("/content/outputs/attention_weights.png", dpi=150)
    plt.show()
else:
    print("No SIMKAP attention artifact captured — run the CV loop first.")

## 20. Channel-level spectral analysis (Figure 11) — model-free sanity check

In [ ]:
SPECTRAL_SAMPLE_SUBJECTS = USABLE_SUBJECTS[:15]
CHANNEL_NAMES = ["AF3", "F7", "F3", "FC5", "T7", "P7", "O1",
                  "O2", "P8", "T8", "FC6", "F4", "F8", "AF4"]


def channel_band_power(root, subj, condition, band):
    raw = load_raw_eeg(root, subj, condition)
    filtered = bandpass_filter(raw)
    normed = minmax_normalize(filtered)
    first_window = normed[:WINDOW_LEN]
    powers = []
    for ch in range(N_CHANNELS):
        _, _, _, bands = spectral_features(first_window[:, ch])
        powers.append(bands[band])
    return np.array(powers)


theta_lo = np.mean([channel_band_power(DATA_ROOT, s, "lo", "theta") for s in SPECTRAL_SAMPLE_SUBJECTS], axis=0)
theta_hi = np.mean([channel_band_power(DATA_ROOT, s, "hi", "theta") for s in SPECTRAL_SAMPLE_SUBJECTS], axis=0)
beta_lo = np.mean([channel_band_power(DATA_ROOT, s, "lo", "beta") for s in SPECTRAL_SAMPLE_SUBJECTS], axis=0)
beta_hi = np.mean([channel_band_power(DATA_ROOT, s, "hi", "beta") for s in SPECTRAL_SAMPLE_SUBJECTS], axis=0)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
x = np.arange(N_CHANNELS)
width = 0.35
axes[0].bar(x - width / 2, theta_lo, width, label="No-Task")
axes[0].bar(x + width / 2, theta_hi, width, label="SIMKAP")
axes[0].set_xticks(x); axes[0].set_xticklabels(CHANNEL_NAMES, rotation=90)
axes[0].set_title("Mean theta-band power by channel"); axes[0].legend()

axes[1].bar(x - width / 2, beta_lo, width, label="No-Task")
axes[1].bar(x + width / 2, beta_hi, width, label="SIMKAP")
axes[1].set_xticks(x); axes[1].set_xticklabels(CHANNEL_NAMES, rotation=90)
axes[1].set_title("Mean beta-band power by channel"); axes[1].legend()

plt.tight_layout()
plt.savefig("/content/outputs/spectral_analysis.png", dpi=150)
plt.show()

print("\nAll analyses complete. Results and figures saved under /content/outputs/")